## Exploración inicial de datos

### By:
Maria Camila Sepulveda Torres

### Date:
2024-08-21

### Description:

Exploración inicial del dataset de admisiones universitarias para comprender su estructura, verificar los tipos de datos, identificar y unificar valores nulos, realizar las conversiones de tipos necesarias y almacenar el dataset resultante en formato Parquet.


In [45]:
# base libraries for data science
from pathlib import Path

import numpy as np
import pandas as pd
import pyarrow as pa

## 💾 Load data

In [46]:
# data directory path
DATA_DIR = Path.cwd().resolve().parents[1] / "data"

admission_df = pd.read_csv(DATA_DIR / "01_raw/Admission_Predict.csv")

In [47]:
admission_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   GRE Score          612 non-null    float64
 1   TOEFL Score        604 non-null    float64
 2   University Rating  617 non-null    float64
 3   SOP                607 non-null    float64
 4   LOR                615 non-null    float64
 5   CGPA               621 non-null    float64
 6   Research           594 non-null    float64
 7   Chance of Admit    623 non-null    float64
dtypes: float64(8)
memory usage: 39.1 KB


In [48]:
admission_df.columns = admission_df.columns.str.strip()

In [49]:
admission_df.columns

Index(['GRE Score', 'TOEFL Score', 'University Rating', 'SOP', 'LOR', 'CGPA',
       'Research', 'Chance of Admit'],
      dtype='str')

In [50]:
admission_df.sample(10)

,GRE Score,TOEFL Score,University Rating,SOP,LOR,CGPA,Research,Chance of Admit
515,316.0,106.0,2.0,2.5,4.0,8.32,0.0,0.72
233,304.0,100.0,2.0,2.5,3.5,8.07,0.0,0.64
460,309.0,105.0,5.0,3.5,3.5,8.56,0.0,0.71
510,307.0,102.0,3.0,3.0,3.0,8.27,0.0,0.73
232,312.0,107.0,2.0,2.5,3.5,8.27,0.0,0.69
350,318.0,107.0,3.0,3.0,3.5,8.27,1.0,0.74
616,318.0,109.0,3.0,3.0,3.0,8.50,0.0,0.67
98,332.0,119.0,4.0,5.0,4.5,9.24,1.0,0.90
587,323.0,112.0,5.0,4.0,4.5,8.78,0.0,0.79
163,317.0,105.0,3.0,3.5,3.0,8.56,0.0,0.68


## 👷 Data preparation or Feature Engineering

### Valores nulos

Se revisan los valores faltantes del dataset con el fin de identificar su presencia y unificar su representación antes de realizar la conversión de los tipos de datos.

In [51]:
admission_df.isnull().sum()

GRE Score            11
TOEFL Score          19
University Rating     6
SOP                  16
LOR                   8
CGPA                  2
Research             29
Chance of Admit       0
dtype: int64

In [52]:
null_representations = ["?", "NA", "N/A", "null", "NULL", ""]

for value in null_representations:
    count = (admission_df == value).sum().sum()
    print(f"{value!r}: {count}")

'?': 0
'NA': 0
'N/A': 0
'null': 0
'NULL': 0
'': 0


Se verificó la presencia de distintas representaciones de valores nulos (`?`, `NA`, `N/A`, `null`, `NULL` y cadenas vacías). No se encontraron representaciones alternativas, por lo que los valores faltantes ya se encuentran unificados y reconocidos por Pandas como `NaN`.

### Categoriqas de las variables

In [53]:
print("University Rating:", sorted(admission_df["University Rating"].dropna().unique()))
print("SOP:", sorted(admission_df["SOP"].dropna().unique()))
print("LOR:", sorted(admission_df["LOR"].dropna().unique()))
print("Research:", sorted(admission_df["Research"].dropna().unique()))

University Rating: [np.float64(1.0), np.float64(2.0), np.float64(3.0), np.float64(4.0), np.float64(5.0)]
SOP: [np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]
LOR: [np.float64(1.0), np.float64(1.5), np.float64(2.0), np.float64(2.5), np.float64(3.0), np.float64(3.5), np.float64(4.0), np.float64(4.5), np.float64(5.0)]
Research: [np.float64(0.0), np.float64(1.0)]



#### Ordinal

- `University Rating`: Calificación de la universidad.
  - Valores de 1 a 5, donde un valor mayor representa una mayor calificación.
- `SOP`: Calificación de la declaración de propósito (Statement of Purpose).
  - Valores de 1 a 5 en incrementos de 0.5.
- `LOR`: Calificación de la carta de recomendación (Letter of Recommendation).
  - Valores de 1 a 5 en incrementos de 0.5.

### Numerical variables

#### Discrete

- `GRE Score`: Puntaje obtenido en el GRE.
- `TOEFL Score`: Puntaje obtenido en el TOEFL.

#### Continuous

- `CGPA`: Promedio académico acumulado.
- `Chance of Admit`: Probabilidad de admisión, representada entre 0 y 1.

### Boolean variables

- `Research`: Indica si el estudiante tiene experiencia en investigación.
  - 0 = No
  - 1 = Sí

In [54]:
cols_categoric = ["University Rating", "SOP", "LOR"]

admission_df[cols_categoric] = admission_df[cols_categoric].astype("category")

In [55]:
admission_df["University Rating"] = pd.Categorical(
    admission_df["University Rating"], categories=[1, 2, 3, 4, 5], ordered=True
)

admission_df["SOP"] = pd.Categorical(
    admission_df["SOP"],
    categories=[1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
    ordered=True,
)

admission_df["LOR"] = pd.Categorical(
    admission_df["LOR"],
    categories=[1.0, 1.5, 2.0, 2.5, 3.0, 3.5, 4.0, 4.5, 5.0],
    ordered=True,
)

In [56]:
cols_numeric_float = ["CGPA", "Chance of Admit"]

admission_df[cols_numeric_float] = admission_df[cols_numeric_float].astype("float")

In [57]:
cols_numeric_int = ["GRE Score", "TOEFL Score"]

admission_df[cols_numeric_int] = admission_df[cols_numeric_int].astype("Int16")

In [58]:
cols_boolean = ["Research"]

admission_df[cols_boolean] = admission_df[cols_boolean].astype("boolean")

In [59]:
admission_df.info()

<class 'pandas.DataFrame'>
RangeIndex: 623 entries, 0 to 622
Data columns (total 8 columns):
 #   Column             Non-Null Count  Dtype   
---  ------             --------------  -----   
 0   GRE Score          612 non-null    Int16   
 1   TOEFL Score        604 non-null    Int16   
 2   University Rating  617 non-null    category
 3   SOP                607 non-null    category
 4   LOR                615 non-null    category
 5   CGPA               621 non-null    float64 
 6   Research           594 non-null    boolean 
 7   Chance of Admit    623 non-null    float64 
dtypes: Int16(2), boolean(1), category(3), float64(2)
memory usage: 16.7 KB


### Guardar el dataframe con los tipos de datos

In [60]:
schema = pa.Table.from_pandas(admission_df).schema

In [61]:
admission_df.to_parquet(
    DATA_DIR / "02_intermediate/admission_type_fixed.parquet",
    index=False,
    schema=schema,
)

## 👷 Data preparation or Feature Engineering

## 📊 Analysis of Results and Conclusions 


La exploración inicial permitió identificar 623 registros y 8 variables. Se encontraron valores nulos en siete variables, los cuales ya estaban representados de forma uniforme como `NaN`.

Se corrigieron los tipos de datos de acuerdo con la naturaleza de cada variable: variables ordinales como categóricas, puntajes como numéricos enteros, `Research` como booleana y las variables continuas como tipo flotante.

Finalmente, el dataset con los tipos corregidos fue almacenado en formato Parquet para ser utilizado en las siguientes etapas del proyecto.
